## Inspect raw data

### Generated Power

In [1]:
import pandas as pd

Path = "C:\\Users\\Felix Privat\\Documents\\ETHachi-TSFM\\data\\wind\\PT15M_20250101_20250201T.parquet"
df = pd.read_parquet(Path)
display(df.head())
print(df.dtypes)
    

region,Federal,Flanders,Wallonia
datetime,,,
2025-01-01 00:00:00+00:00,2037.98,1214.15,595.39
2025-01-01 00:15:00+00:00,2075.45,1248.95,602.60
2025-01-01 00:30:00+00:00,2074.44,1252.67,627.67
2025-01-01 00:45:00+00:00,2054.38,1222.53,597.27
2025-01-01 01:00:00+00:00,2087.39,1273.62,671.71


region
Federal     float64
Flanders    float64
Wallonia    float64
dtype: object


### Weather

In [2]:
import pandas as pd 

weather_path = "C:\\Users\\Felix Privat\\Documents\\ETHachi-TSFM\\data\\weather\\gen_weather_dwd.parquet"

df = pd.read_parquet(weather_path).set_index("timestamp") 

y = df["measured"]                  # target [MW] 

X = df.drop(columns="measured")     # weather covariates 

display(df.head())
print(df.dtypes)

FileNotFoundError: C:/Users/Felix Privat/Documents/ETHachi-TSFM/data/weather/gen_weather_dwd.parquet

In [ ]:
import pandas as pd 

weather_path = "C:\\Users\\Felix Privat\\Documents\\ETHachi-TSFM\\data\\weather\\PT1H_20240101_20240131T.parquet"

df = pd.read_parquet(weather_path)

display(df.head())
print(df.dtypes)

,cloud_cover_low,cloud_cover_mid,cloud_cover_high,direct_radiation,diffuse_radiation,direct_normal_irradiance
datetime,,,,,,
2024-01-01 00:00:00+00:00,100,91,14,0.0,0.0,0.0
2024-01-01 01:00:00+00:00,100,97,99,0.0,0.0,0.0
2024-01-01 02:00:00+00:00,35,1,23,0.0,0.0,0.0
2024-01-01 03:00:00+00:00,29,1,0,0.0,0.0,0.0
2024-01-01 04:00:00+00:00,46,6,17,0.0,0.0,0.0


cloud_cover_low               int64
cloud_cover_mid               int64
cloud_cover_high              int64
direct_radiation            float64
diffuse_radiation           float64
direct_normal_irradiance    float64
dtype: object


### Testing functions

In [ ]:
from src.config import DATA_USE_DWD
from src.features.features import merge_dwd_weather

import pandas as pd 

weather_path = "C:\\Users\\Felix Privat\\Documents\\ETHachi-TSFM\\data\\weather\\PT1H_20240101_20240131T.parquet"

df = pd.read_parquet(weather_path)

dwd_path = "C:\\Users\\Felix Privat\\Documents\\ETHachi-TSFM\\data\\weather\\gen_weather_dwd.parquet"

df_dwd = merge_dwd_weather(df, dwd_path)

merge_df = merge_dwd_weather(df, dwd_path)

display(merge_df.head())

2026-10-16 11:28:30,554 | WARNING  | features | Dropped 70179 of 73152 rows with missing values. Missing per column: {'cloud_cover_low': 70179, 'cloud_cover_mid': 70179, 'cloud_cover_high': 70179, 'direct_radiation': 70179, 'diffuse_radiation': 70179, 'direct_normal_irradiance': 70179}
2026-10-16 11:28:30,631 | WARNING  | features | Dropped 70179 of 73152 rows with missing values. Missing per column: {'cloud_cover_low': 70179, 'cloud_cover_mid': 70179, 'cloud_cover_high': 70179, 'direct_radiation': 70179, 'diffuse_radiation': 70179, 'direct_normal_irradiance': 70179}


,cloud_cover_low,cloud_cover_mid,cloud_cover_high,direct_radiation,diffuse_radiation,direct_normal_irradiance,temperature_2m_mean,temperature_2m_std,relative_humidity_2m_mean,relative_humidity_2m_std,...,wind_direction_10m_mean,wind_direction_10m_std,wind_direction_80m_mean,wind_direction_80m_std,wind_direction_120m_mean,wind_direction_120m_std,wind_direction_180m_mean,wind_direction_180m_std,wind_gusts_10m_mean,wind_gusts_10m_std
datetime,,,,,,,,,,,,,,,,,,,,,
2024-01-01 00:00:00+00:00,100.0,91.0,14.00,0.0,0.0,0.0,9.681818,0.106876,82.545456,2.478188,...,237.786575,2.328713,238.594269,2.355603,239.065781,2.364003,239.647308,2.229698,76.064728,2.674927
2024-01-01 00:15:00+00:00,100.0,92.5,35.25,0.0,0.0,0.0,9.724772,0.099530,81.322731,2.456124,...,239.054504,2.255149,239.805069,2.234068,240.241608,2.213781,240.760925,2.082111,75.460907,2.368973
2024-01-01 00:30:00+00:00,100.0,94.0,56.50,0.0,0.0,0.0,9.767727,0.092184,80.100006,2.434061,...,240.322418,2.181586,241.015869,2.112534,241.417450,2.063560,241.874542,1.934523,74.857086,2.063018
2024-01-01 00:45:00+00:00,100.0,95.5,77.75,0.0,0.0,0.0,9.810682,0.084838,78.877274,2.411997,...,241.590347,2.108022,242.226669,1.991000,242.593277,1.913338,242.988144,1.786936,74.253273,1.757064
2024-01-01 01:00:00+00:00,100.0,97.0,99.00,0.0,0.0,0.0,9.853637,0.077492,77.654549,2.389934,...,242.858276,2.034459,243.437469,1.869465,243.769104,1.763116,244.101761,1.639348,73.649452,1.451109


## Test wind region weather data fetch

In [1]:
import config
from src.http_client import HttpClient
from src.datasource import generate_wind_weather_samples, load_wind_weather_statistics

# 1) Sample points: Elia wind production share per region -> capacity-weighted wind farms
with HttpClient(config.ELIA_SOURCES["wind"]["base_url"]) as elia:
    points = generate_wind_weather_samples(elia)
print(len(points))
print(points["region"].value_counts())
display(points.head())

# 2) Open-Meteo fetch (chunked + cached) and statistical reduction -> data/wind-region/*.parquet
src = config.OPENMETEO_SOURCES["weather"]
with HttpClient(src["base_url"], timeout=120) as om:
    wind_stats = load_wind_weather_statistics(
        "wind_weather", om, points, src["hourly_wind"], config.DATA_START, config.DATA_END
    )
print(wind_stats.shape)
display(wind_stats.head())
wind_stats[["wind_speed_120m_mean", "wind_speed_120m_std"]].plot(figsize=(12, 3))


2026-11-13 18:44:51,793 | INFO     | datasource | Fetching 2025 wind region production from the Elia API
2026-11-13 18:44:52,418 | INFO     | datasource | Cached wind region production shares to C:\Users\Felix Privat\Documents\ETHachi-TSFM\data\wind-region\production_share_2025.csv


50
region
Federal     32
Flanders    11
Wallonia     7
Name: count, dtype: int64


,latitude,longitude,region
0,51.660100,2.800040,Federal
1,51.566647,3.000000,Federal
2,51.720000,2.740000,Federal
3,51.513500,2.991510,Federal
4,51.686111,2.759444,Federal


2026-11-13 18:44:52,468 | INFO     | datasource | Fetching wind_weather for 50 points from the Open-Meteo API
2026-11-13 18:44:52,470 | INFO     | datasource | Fetching points 0-9 of 50
2026-11-13 18:46:24,923 | INFO     | datasource | Fetching points 10-19 of 50


KeyboardInterrupt: 

In [3]:
display(points)

,latitude,longitude,region
0,51.660100,2.800040,Federal
1,51.566647,3.000000,Federal
2,51.720000,2.740000,Federal
3,51.513500,2.991510,Federal
4,51.686111,2.759444,Federal
5,51.686111,2.759444,Federal
6,51.575700,2.918800,Federal
7,51.513500,2.991510,Federal
8,51.566647,3.000000,Federal
9,51.566647,3.000000,Federal


from 